# Lesson 08
Run lessons 1–7 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第8课 强化学习与序贯配置（TPU训练＋SB3基线）

目标：定义观察、动作、奖励和episode，让PPO在第六课相同资产、总多头0.8、单股0.15、行业0.35与换手上限0.25下选择配置。动作从现金、等权、逆波动、最小方差、状态过滤5个已知可行组合中选择，执行端进一步处理真实漂移和换手；所有策略最终输出target_weight并由第二课账户重放。

状态包含过去动量、波动、趋势、统计套利残差、因子摘要、当前仓位与回撤；不含未来收益或未成熟标签。数值归一化仅fit2014—2018。未来价格仅在step转移后产生净收益和奖励。

奖励=100×log(Vnext/V)−λ×max(Dnext−0.10,0)，V已扣一次真实交易费用。回撤惩罚是学习目标，不再次从现金扣钱。现金可能提高奖励但降低资本参与率，必须对照平均敞口、换手和净收益。历史回放不响应智能体，不能当真实在线学习。

先check_env、随机动作和固定权重手工核账。Stable-Baselines3 PPO作为主机参考；另实现离散动作PPO的JAX梯度更新，在实际TPU上训练，保留裁剪目标、GAE、价值损失和熵项。两种实现各5随机种子、固定训练步数；λ=0作为TPU奖励消融。验证2019—2020选择模型家族；2021—2022为本项目已观察开发评估，冻结选择后一次报告，不声称全新测试。

episode到数据边界为terminated，到人为长度为truncated；GAE对truncated保留末态bootstrap，对terminated归零。模型检查点保存为内存字节，最终打入云端项目包，不写本地文件。多种子报告中位数、四分位差、失败和不及规则基线次数。

参考：[Gymnasium自定义环境](https://gymnasium.farama.org/introduction/create_custom_env/)、[SB3评估和多种子](https://stable-baselines3.readthedocs.io/en/v2.7.0/guide/rl_tips.html)。


In [ ]:
# 08.01 因果状态、可行候选动作与同账本一步转移
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
import torch
SEEDS8=[11,22,33,44,55];BUDGET8=8192
DATES8=EXEC6;PREV8=PREV6
FEATURE8=pd.DataFrame(index=DATES8)
for c in ['hist_vol','ewma_vol','trend']:
    FEATURE8[c]=PREV8.map(SIG6[c])
FEATURE8['pair_z']=PREV8.map(l2_features.z)
FEATURE8['market_ret20']=PREV8.map(CLOSE.SPY.pct_change(20,fill_method=None))
for f in FACTOR_NAMES:
    daily=PANEL.groupby('date')[f].mean();FEATURE8['cross_'+f]=PREV8.map(daily)
FEATURE8=FEATURE8.replace([np.inf,-np.inf],np.nan)
TRAIN8=np.flatnonzero(DATES8<TRAIN_END);VALID8=np.flatnonzero((DATES8>=TRAIN_END)&(DATES8<VALID_END));TEST8=np.flatnonzero(DATES8>=VALID_END)
# 训练最后一个转移的估值开盘也必须在验证前，保守剔除边界。
END8=pd.Series(OPEN.index,index=OPEN.index).shift(-1).reindex(DATES8)
TRAIN8=np.flatnonzero((DATES8<TRAIN_END)&(END8.to_numpy()<TRAIN_END.to_datetime64()))
VALID8=np.flatnonzero((DATES8>=TRAIN_END)&(DATES8<VALID_END)&(END8.to_numpy()<VALID_END.to_datetime64()))
MED8=FEATURE8.iloc[TRAIN8].median();MEAN8=FEATURE8.iloc[TRAIN8].fillna(MED8).mean();STD8=FEATURE8.iloc[TRAIN8].fillna(MED8).std().clip(lower=1e-6)
X8=((FEATURE8.fillna(MED8)-MEAN8)/STD8).clip(-10,10).to_numpy(np.float32)
SLEEVES8=['cash','equal','inverse_vol','shrink_minvar','state_filtered']
A8=np.stack([np.zeros((len(DATES8),len(ASSETS)))]+[DESIRED6[n].reindex(DATES8).to_numpy() for n in SLEEVES8[1:]],axis=1).astype(np.float32)
RETURN8=(OPEN[ASSETS].shift(-1)/OPEN[ASSETS]-1).reindex(DATES8).to_numpy(np.float64)
assert np.isfinite(X8).all() and np.isfinite(RETURN8).all()
def step_account8(weight,wealth,peak,desired,ret,cost=.0005,penalty=.05):
    old=np.asarray(weight,float);desired=np.asarray(desired,float)
    repair=min(1.,.8/max(old.sum(),1e-12),.15/max(old.max(),1e-12),*[.35/max(old[j].sum(),1e-12) for j in GROUP6.values()])
    safe=old*repair
    def trade(w):
        post=1.
        for _ in range(20):post=1-cost*np.abs(w*post-old).sum()
        delta=w*post-old;return post,delta
    post,delta=trade(desired);turn=np.abs(delta).sum()
    if turn>.25:
        lo,hi=0.,1.
        for _ in range(28):
            mid=(lo+hi)/2;p,d=trade(safe+mid*(desired-safe))
            if np.abs(d).sum()<=.25:lo=mid
            else:hi=mid
        desired=safe+lo*(desired-safe);post,delta=trade(desired)
    fee=cost*np.abs(delta).sum();cash=1-(desired*post).sum()-fee
    end_values=desired*post*(1+ret);growth=cash+end_values.sum();new_wealth=wealth*growth;new_peak=max(peak,new_wealth)
    dd=1-new_wealth/new_peak;reward=100*np.log(max(growth,1e-12))-penalty*max(dd-.10,0)
    return end_values/growth,new_wealth,new_peak,reward,desired,fee,np.abs(delta).sum()
class PortfolioEnv8(gym.Env):
    metadata={'render_modes':[]}
    def __init__(self,indices,training=False,penalty=.05,horizon=128):
        self.indices=np.asarray(indices);self.training=training;self.penalty=penalty;self.horizon=horizon
        self.action_space=spaces.Discrete(len(SLEEVES8));self.observation_space=spaces.Box(-np.inf,np.inf,shape=(X8.shape[1]+len(ASSETS)+1,),dtype=np.float32)
    def observation(self):return np.r_[X8[self.i],self.weight,1-self.wealth/self.peak].astype(np.float32)
    def reset(self,seed=None,options=None):
        super().reset(seed=seed);start=int(self.np_random.integers(0,max(1,len(self.indices)-self.horizon))) if self.training else 0
        self.local=start;self.i=int(self.indices[start]);self.steps=0;self.weight=np.zeros(len(ASSETS));self.wealth=1.;self.peak=1.;self.history=[]
        return self.observation(),{}
    def step(self,action):
        d=DATES8[self.i];oldwealth=self.wealth
        self.weight,self.wealth,self.peak,reward,target,fee,turn=step_account8(self.weight,self.wealth,self.peak,A8[self.i,int(action)],RETURN8[self.i],penalty=self.penalty)
        info={'date':d,'target_weight':target.copy(),'fee_fraction':fee,'turnover':turn,'wealth':self.wealth,'net_return':self.wealth/oldwealth-1}
        self.history.append(info);self.steps+=1;self.local+=1
        terminated=self.local>=len(self.indices) or self.wealth<=0
        truncated=bool(self.training and self.steps>=self.horizon and not terminated)
        if not terminated:self.i=int(self.indices[self.local])
        obs=self.observation()
        return obs,float(reward),bool(terminated),truncated,info
ENV8=PortfolioEnv8(TRAIN8,True);check_env(ENV8,warn=True)
obs,_=ENV8.reset(seed=11)
for _ in range(20):
    obs,reward,terminated,truncated,info=ENV8.step(ENV8.action_space.sample())
    assert np.isfinite(obs).all() and np.isfinite(reward)
    if terminated or truncated:break
print('08.01 PASS',X8.shape,A8.shape,'normalizer fit cutoff',str(DATES8[TRAIN8[-1]]),'actions',SLEEVES8)


In [ ]:
# 08.02 SB3五种子基线、冻结评估与内存检查点
SB3_MODELS8={};CHECKPOINTS8={};TRAIN_LOG8=[]
def evaluate_policy8(policy,indices,penalty=.05):
    env=PortfolioEnv8(indices,False,penalty=penalty);obs,_=env.reset();actions=[];done=False
    while not done:
        action=int(policy(obs));actions.append(action);obs,r,term,trunc,info=env.step(action);done=term or trunc
    target=pd.DataFrame([x['target_weight'] for x in env.history],index=[x['date'] for x in env.history],columns=ASSETS)
    ledger,trades=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
    assert np.allclose(ledger.wealth,[h['wealth'] for h in env.history],rtol=2e-5,atol=2e-6)
    return ledger,target,np.asarray(actions)
SB3_EVAL8={};BASELINE_EVAL8={};SEED_RESULTS8=[]
for name,a in [('cash',0),('equal',1),('inverse_vol',2),('rule_risk',4)]:
    for fold,indices in [('validation',VALID8),('development_eval',TEST8)]:
        x,t,act=evaluate_policy8(lambda obs,a=a:a,indices);BASELINE_EVAL8[(name,fold)]=(x,t,act)
        SEED_RESULTS8.append({'implementation':'rule','penalty':.05,'seed':-1,'model':name,'fold':fold,**ledger_stats(x),'avg_gross':x.gross_exposure.mean()})
torch.set_num_threads(2)
for seed in SEEDS8:
    started=time.perf_counter()
    try:
        model=PPO('MlpPolicy',PortfolioEnv8(TRAIN8,True,penalty=.05,horizon=128),seed=seed,device='cpu',n_steps=128,batch_size=64,
          n_epochs=4,learning_rate=3e-4,gamma=.99,gae_lambda=.95,clip_range=.2,ent_coef=.01,vf_coef=.5,
          policy_kwargs={'net_arch':dict(pi=[32,32],vf=[32,32])},verbose=0)
        model.learn(total_timesteps=BUDGET8);SB3_MODELS8[seed]=model
        buffer=io.BytesIO();model.save(buffer);CHECKPOINTS8['SB3_seed_'+str(seed)+'.zip']=buffer.getvalue()
        for fold,indices in [('validation',VALID8),('development_eval',TEST8)]:
            x,t,act=evaluate_policy8(lambda obs,m=model:int(m.predict(obs,deterministic=True)[0]),indices)
            SB3_EVAL8[(seed,fold)]=(x,t,act)
            SEED_RESULTS8.append({'implementation':'SB3_CPU','penalty':.05,'seed':seed,'model':'PPO','fold':fold,**ledger_stats(x),'avg_gross':x.gross_exposure.mean()})
        TRAIN_LOG8.append({'implementation':'SB3_CPU','seed':seed,'timesteps':int(model.num_timesteps),'status':'completed','seconds':time.perf_counter()-started})
    except Exception as e:TRAIN_LOG8.append({'implementation':'SB3_CPU','seed':seed,'status':'failed','error':str(e)})
    print('SB3 seed',seed,TRAIN_LOG8[-1]['status'])
assert len(SB3_MODELS8)==5,'Some seed failed: retain failure log; do not silently replace seeds'
display(pd.DataFrame(SEED_RESULTS8));display(pd.DataFrame(TRAIN_LOG8));print('08.02 PASS: five reference seeds, exact replay, in-memory checkpoints')


In [ ]:
# 08.03 JAX离散PPO：实际TPU梯度、GAE与五种子奖励消融
from functools import partial
OBS_DIM8=X8.shape[1]+len(ASSETS)+1
with jax.default_device(TPU):
    def init_ppo8(seed):
        keys=jax.random.split(jax.random.PRNGKey(seed),4)
        return {'W1':jax.random.normal(keys[0],(OBS_DIM8,32))*np.sqrt(2/OBS_DIM8),'b1':jnp.zeros(32),
         'W2':jax.random.normal(keys[1],(32,32))*np.sqrt(2/32),'b2':jnp.zeros(32),
         'Wp':jax.random.normal(keys[2],(32,5))*.01,'bp':jnp.zeros(5),
         'Wv':jax.random.normal(keys[3],(32,1))*.1,'bv':jnp.zeros(1)}
@jax.jit
def actor8(params,obs):
    h=jnp.tanh(obs@params['W1']+params['b1']);h=jnp.tanh(h@params['W2']+params['b2'])
    return h@params['Wp']+params['bp'],(h@params['Wv']+params['bv']).squeeze(-1)
def ppo_loss8(params,batch):
    obs,act,old_log,adv,target=batch;logits,value=actor8(params,obs);logp=jax.nn.log_softmax(logits);new_log=jnp.take_along_axis(logp,act[:,None],axis=1).squeeze(1)
    ratio=jnp.exp(new_log-old_log);a=(adv-adv.mean())/(adv.std()+1e-8)
    policy=-jnp.mean(jnp.minimum(ratio*a,jnp.clip(ratio,.8,1.2)*a));v_loss=.5*jnp.mean((value-target)**2)
    entropy=-jnp.mean(jnp.sum(jnp.exp(logp)*logp,axis=1))
    return policy+.5*v_loss-.01*entropy,(policy,v_loss,entropy,jnp.mean((ratio-1)-jnp.log(ratio)),jnp.mean(jnp.abs(ratio-1)>.2))
@jax.jit
def adam_update8(params,m,v,step,batch):
    (loss,diagnostics),grad=jax.value_and_grad(ppo_loss8,has_aux=True)(params,batch)
    magnitude=jnp.sqrt(sum(jnp.sum(g*g) for g in jax.tree_util.tree_leaves(grad)))
    grad=jax.tree_util.tree_map(lambda g:g*jnp.minimum(1.,.5/(magnitude+1e-8)),grad)
    m=jax.tree_util.tree_map(lambda x,g:.9*x+.1*g,m,grad);v=jax.tree_util.tree_map(lambda x,g:.999*x+.001*g*g,v,grad)
    step=step+1
    params=jax.tree_util.tree_map(lambda p,a,b:p-3e-4*(a/(1-.9**step))/(jnp.sqrt(b/(1-.999**step))+1e-8),params,m,v)
    return params,m,v,step,loss,diagnostics
JAX_MODELS8={};JAX_EVAL8={};PPO_DIAGNOSTICS8=[]
for penalty in [.05,0.]:
    for seed in SEEDS8:
        started=time.perf_counter();rng=np.random.default_rng(seed);env=PortfolioEnv8(TRAIN8,True,penalty=penalty,horizon=128)
        try:
            with jax.default_device(TPU):
                params=init_ppo8(seed);m=jax.tree_util.tree_map(jnp.zeros_like,params);v=jax.tree_util.tree_map(jnp.zeros_like,params);step=jnp.array(0)
            for update in range(BUDGET8//128):
                obs,_=env.reset(seed=seed*1000+update);O=[];ACT=[];LP=[];V=[];NEXT=[];REW=[];TERM=[];CUT=[]
                for k in range(128):
                    logits,value=actor8(params,jax.device_put(np.asarray(obs,np.float32),TPU));logit=np.asarray(logits);prob=np.exp(logit-logit.max());prob=prob/prob.sum()
                    action=int(rng.choice(5,p=prob));nextobs,reward,term,trunc,info=env.step(action)
                    _,nextvalue=actor8(params,jax.device_put(nextobs,TPU))
                    O.append(obs);ACT.append(action);LP.append(np.log(max(prob[action],1e-12)));V.append(float(value));NEXT.append(float(nextvalue));REW.append(reward);TERM.append(term);CUT.append(trunc)
                    obs=nextobs
                    if term or trunc:break
                advantages=np.zeros(len(REW),np.float32);carry=0.
                for k in reversed(range(len(REW))):
                    delta=REW[k]+.99*NEXT[k]*(1-int(TERM[k]))-V[k]
                    carry=delta+.99*.95*(1-int(TERM[k] or CUT[k]))*carry;advantages[k]=carry
                batch=tuple(jax.device_put(np.asarray(z,dtype=d),TPU) for z,d in zip([O,ACT,LP,advantages,advantages+np.asarray(V)], [np.float32,np.int32,np.float32,np.float32,np.float32]))
                assert len(REW)==128,'fixed rollout budget changed unexpectedly'
                for epoch in range(4):params,m,v,step,loss,diag=adam_update8(params,m,v,step,batch)
                if update%16==0 or update==BUDGET8//128-1:
                    loss.block_until_ready();PPO_DIAGNOSTICS8.append({'penalty':penalty,'seed':seed,'update':update,'loss':float(loss),'policy_loss':float(diag[0]),'value_loss':float(diag[1]),'entropy':float(diag[2]),'approx_KL':float(diag[3]),'clip_fraction':float(diag[4])})
            assert all(np.isfinite(np.asarray(p)).all() for p in jax.tree_util.tree_leaves(params))
            JAX_MODELS8[(penalty,seed)]=params
            buffer=io.BytesIO();np.savez_compressed(buffer,**{k:np.asarray(v) for k,v in params.items()});CHECKPOINTS8['JAX_penalty_'+str(penalty)+'_seed_'+str(seed)+'.npz']=buffer.getvalue()
            for fold,indices in [('validation',VALID8),('development_eval',TEST8)]:
                x,t,act=evaluate_policy8(lambda obs,p=params:int(np.asarray(actor8(p,jax.device_put(obs,TPU))[0]).argmax()),indices,penalty=penalty)
                JAX_EVAL8[(penalty,seed,fold)]=(x,t,act)
                SEED_RESULTS8.append({'implementation':'JAX_TPU','penalty':penalty,'seed':seed,'model':'PPO','fold':fold,**ledger_stats(x),'avg_gross':x.gross_exposure.mean()})
            TRAIN_LOG8.append({'implementation':'JAX_TPU','penalty':penalty,'seed':seed,'timesteps':BUDGET8,'updates':int(step),'device':str(TPU),'status':'completed','seconds':time.perf_counter()-started})
        except Exception as e:
            TRAIN_LOG8.append({'implementation':'JAX_TPU','penalty':penalty,'seed':seed,'status':'failed','error':str(e)});print('JAX failure',penalty,seed,str(e));raise
        print('JAX TPU',penalty,seed,'completed',round(time.perf_counter()-started,1),'seconds',flush=True)
assert len(JAX_MODELS8)==10
print('Gradient arrays actually on',next(iter(JAX_MODELS8.values()))['W1'].devices());display(pd.DataFrame(PPO_DIAGNOSTICS8).tail(10));display(pd.DataFrame(TRAIN_LOG8))
print('08.03 PASS: 10 TPU checkpoints; fixed seeds and budgets; truncation bootstrap explicit')


In [ ]:
# 08.04 多种子冻结选择、奖励消融及环境漏洞审计
RESULT8=pd.DataFrame(SEED_RESULTS8)
PPO_RESULTS8=RESULT8[RESULT8.model.eq('PPO')].copy();PPO_RESULTS8['family']=PPO_RESULTS8.implementation+'_lambda_'+PPO_RESULTS8.penalty.astype(str)
SUMMARY8=PPO_RESULTS8.groupby(['fold','family']).agg(median_return=('net_return','median'),q25_return=('net_return',lambda x:x.quantile(.25)),
 q75_return=('net_return',lambda x:x.quantile(.75)),median_Sharpe=('Sharpe_0rf','median'),median_drawdown=('max_drawdown','median'),avg_gross=('avg_gross','median'),runs=('seed','size'))
display(SUMMARY8);display(RESULT8[RESULT8.implementation.eq('rule')])
family_validation=SUMMARY8.xs('validation').median_Sharpe
SELECTED_FAMILY8=family_validation.sort_values(ascending=False,kind='stable').index[0]
SELECTED_IMPL8='JAX_TPU' if SELECTED_FAMILY8.startswith('JAX') else 'SB3_CPU'
SELECTED_PENALTY8=float(SELECTED_FAMILY8.rsplit('_',1)[1])
print('Family selected on validation median Sharpe only:',SELECTED_FAMILY8)
FAILURES8=[]
for fold in ['validation','development_eval']:
    baseline=float(RESULT8[(RESULT8.model=='rule_risk')&(RESULT8.fold==fold)].net_return.iloc[0])
    for family,g in PPO_RESULTS8[PPO_RESULTS8.fold.eq(fold)].groupby('family'):
        FAILURES8.append({'fold':fold,'family':family,'n':len(g),'negative_returns':int((g.net_return<0).sum()),'below_rule_risk':int((g.net_return<=baseline).sum())})
display(pd.DataFrame(FAILURES8));print('训练运行失败',sum(r['status']=='failed' for r in TRAIN_LOG8),'；性能不及规则与程序失败分开报告。')
# 共资本合成：平均的是同资产目标权重，再用一个账户记账，不平均独立账户净值。
ENSEMBLE8={}
for fold in ['validation','development_eval']:
    targets=[SB3_EVAL8[(s,fold)][1] if SELECTED_IMPL8=='SB3_CPU' else JAX_EVAL8[(SELECTED_PENALTY8,s,fold)][1] for s in SEEDS8]
    proposed=sum(targets)/len(targets);actual,audit=limited_targets(OPEN[ASSETS],proposed)
    ledger,trades=l2_simulate_multi(OPEN[ASSETS],actual,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
    ENSEMBLE8[fold]=(ledger,actual,audit);print('Selected family ensemble',fold,ledger_stats(ledger))
L8_CHECKS=[]
def check8(name,ok):
    assert bool(ok),name;L8_CHECKS.append({'check':name,'status':'PASS'})
check8('SB3 five completed seeds',len(SB3_MODELS8)==5)
check8('TPU five seeds per reward variant',len(JAX_MODELS8)==10 and all(v['W1'].devices()=={TPU} for v in JAX_MODELS8.values()))
check8('fixed actual training budget',all(r.get('timesteps')==BUDGET8 for r in TRAIN_LOG8 if r['status']=='completed'))
check8('normalizer train median',np.allclose(MED8,FEATURE8.iloc[TRAIN8].median(),equal_nan=True))
check8('train end before validation',(END8.iloc[TRAIN8]<TRAIN_END).all())
check8('validation end before evaluation',(END8.iloc[VALID8]<VALID_END).all())
# 人工价格突变只能影响转移，不应改变当前观察。
env=PortfolioEnv8(VALID8,False);o1,_=env.reset();i=env.i;saved=RETURN8[i].copy();RETURN8[i]=saved+.05;o2=env.observation();RETURN8[i]=saved
check8('future return absent from current state',np.array_equal(o1,o2))
# 手算无波动买入：费用只扣一次，扣费后权益=1/(1+c*targetgross)。
_,wealth,_,reward,target,fee,turn=step_account8(np.zeros(12),1.,1.,np.ones(12)*.2/12,np.zeros(12),penalty=0)
check8('manual self financing fee',np.isclose(wealth,1/(1+.0005*.2),atol=1e-9) and np.isclose(fee,1-wealth,atol=1e-9))
check8('reward is net log return',np.isclose(reward,100*np.log(wealth)))
short_env=PortfolioEnv8(TRAIN8,True,horizon=2);short_env.reset(seed=3);short_env.step(1);_,_,term,trunc,_=short_env.step(1)
check8('artificial horizon is truncation',trunc and not term)
last_env=PortfolioEnv8(TRAIN8[-2:],False);last_env.reset();last_env.step(1);_,_,term,trunc,_=last_env.step(1)
check8('data boundary is termination',term and not trunc)
for family,collection in [('SB3',SB3_EVAL8),('JAX',JAX_EVAL8)]:
    for key,(x,t,a) in collection.items():
        check8(family+str(key)+' accounting',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost))
        check8(family+str(key)+' constraints',(t>=-1e-7).all().all() and t.sum(axis=1).max()<=.800001 and t.max().max()<=.150001 and all(t.iloc[:,j].sum(axis=1).max()<=.350001 for j in GROUP6.values()))
        check8(family+str(key)+' no invalid action',np.isin(a,np.arange(5)).all())
        check8(family+str(key)+' turnover bound',x.turnover.max()<=.25001)
L8_VALIDATION=pd.DataFrame(L8_CHECKS);display(L8_VALIDATION)
fig,ax=plt.subplots(1,2,figsize=(13,4));dev=PPO_RESULTS8[PPO_RESULTS8.fold.eq('development_eval')]
for family,g in dev.groupby('family'):ax[0].scatter([family]*len(g),g.net_return);ax[0].tick_params(axis='x',rotation=20)
ax[0].set_title('All seeds: already observed development evaluation')
for n in ['equal','rule_risk']:
    x=BASELINE_EVAL8[(n,'development_eval')][0];ax[1].plot(x.index,x.wealth,label=n)
x=ENSEMBLE8['development_eval'][0];ax[1].plot(x.index,x.wealth,label='validation selected family target ensemble');ax[1].legend();plt.tight_layout();plt.show()
L8_CARD={'lesson':8,'status':'verified','checks':len(L8_CHECKS),'seeds':SEEDS8,'steps_per_seed':BUDGET8,
 'checkpoints':len(CHECKPOINTS8),'selected_on_validation':SELECTED_FAMILY8,'normalizer_fit_max':str(DATES8[TRAIN8[-1]]),
 'limits':['small training budget','discrete feasible sleeve space','fixed survivors','historical replay, not market interaction','already observed development windows','no performance guarantee']}
print('L8 FINAL PASS',json.dumps(L8_CARD,ensure_ascii=False))
